# Your first compiled Python loop

**Core notebook, about 25 minutes.** We will time a Python loop, ask Numba to make a faster version, check that the answer is unchanged, and time it again.

The important habit is: **check, time, change, check again, then time again.**

In [ ]:
import numpy as np
import numba
from numba import jit

print("NumPy", np.__version__)
print("Numba", numba.__version__)

## 1. Time the current Python version

This function contains an explicit loop. It is intentionally written as clear Python before we optimize it.

In [ ]:
def diagonal_trace_python(a):
    trace = 0.0
    for i in range(a.shape[0]):
        trace += np.tanh(a[i, i])
    return trace

x = np.arange(1_000_000, dtype=np.float64).reshape(1000, 1000)
python_result = diagonal_trace_python(x)
python_time = %timeit -o -n 10 -r 3 diagonal_trace_python(x)

## 2. Add `@jit`, call once, and check the answer

`@jit` asks Numba to compile the function, which means making a machine-code version that can run faster. Numba does this work during the first call, so time later calls instead of the first one.

In [ ]:
diagonal_trace_numba = jit(diagonal_trace_python)

# First call: make the compiled version and return an answer.
compiled_result = diagonal_trace_numba(x)

# Correctness comes before speed.
np.testing.assert_allclose(compiled_result, python_result, rtol=1e-12)

# The compiled version already exists, so this is a fair timing.
numba_time = %timeit -o -n 10 -r 3 diagonal_trace_numba(x)
print(f"Speedup after the first call: {python_time.average / numba_time.average:.1f}x")

## 3. Compare with NumPy

Numba is not a replacement for clear NumPy. If NumPy can express the work in one clear line, time both versions.

In [ ]:
def diagonal_trace_numpy(a):
    return np.tanh(np.diagonal(a)).sum()

numpy_result = diagonal_trace_numpy(x)
np.testing.assert_allclose(numpy_result, python_result, rtol=1e-12)
numpy_time = %timeit -o -n 10 -r 3 diagonal_trace_numpy(x)

print(f"Python: {python_time.average * 1e3:.3f} ms")
print(f"Numba:  {numba_time.average * 1e3:.3f} ms")
print(f"NumPy:  {numpy_time.average * 1e3:.3f} ms")

## Your turn: sum of squares

**12 minutes.** Write a function that computes the sum of squares using a Python `for` loop. Then:

1. Compile it with `jit`.
2. Check it against `np.sum(values ** 2)`.
3. Warm up before timing.
4. Time Numba and NumPy.
5. Explain the result in one sentence.

```python
def sum_of_squares(values):
    total = 0.0
    # Add your loop.
    return total
```

Blue sticky note means you need help. Yellow means you are ready to discuss.

In [ ]:
# Write and test your solution here before opening the solution cell below.

<details><summary>Solution and discussion</summary>

Run the next cell only after attempting the exercise. NumPy may be as fast or faster for this simple sum. Numba is especially useful when one loop combines several calculations or conditions and would be awkward to write as one NumPy expression.

</details>

In [ ]:
def sum_of_squares_python(values):
    total = 0.0
    for value in values:
        total += value * value
    return total

sum_of_squares_numba = jit(sum_of_squares_python)
rng = np.random.default_rng(2026)
values = rng.random(1_000_000)

expected = np.sum(values ** 2)
actual = sum_of_squares_numba(values)  # compile and warm up
np.testing.assert_allclose(actual, expected, rtol=1e-12)

exercise_numba_time = %timeit -o -n 10 -r 3 sum_of_squares_numba(values)
exercise_numpy_time = %timeit -o -n 10 -r 3 np.sum(values ** 2)
print(f"Numba: {exercise_numba_time.average * 1e6:.1f} us")
print(f"NumPy: {exercise_numpy_time.average * 1e6:.1f} us")

## Takeaway

- Time the current version so you know where the program is slow.
- Prefer clear NumPy when it already expresses the work well.
- Try Numba for slow numerical Python loops.
- Check the answer, call the compiled function once, and then time later calls.

Optional next step: [`1_numpy.ipynb`](1_numpy.ipynb).